In [33]:
import os
import torch
import torchvision.transforms as T
import torch.nn as nn
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
import numpy as np
from tqdm import tqdm
import utils


from pprint import pprint
from torch.utils.data import DataLoader

In [34]:
architecture = "UNET"
encoder = "resnet34"
preprocessing = smp.encoders.get_preprocessing_fn(encoder)
print(preprocessing)

functools.partial(<function preprocess_input at 0x7f561c456c20>, input_space='RGB', input_range=[0, 1], mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])


In [35]:
# transform like preprocess required by the network
tf = T.Compose([
    T.Resize(256),
    T.ConvertImageDtype(torch.float),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [36]:
from torchvision.datasets import OxfordIIITPet

train_dataset = OxfordIIITPet(root = "dataset", split="trainval", transforms=tf, download=True)
test_dataset = OxfordIIITPet(root = "dataset", split="test", download=True)
train_dataset, valid_dataset = torch.utils.data.random_split(train_dataset, [3380, 300])

print(len(train_dataset))
print(len(test_dataset))
print(len(valid_dataset))

3380
3669
300


In [37]:
# Hyperparameters:
LEARNING_RATE = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
NUM_EPOCHS = 5
LOAD_MODEL = False
NUM_WORKERS = os.cpu_count()


In [38]:
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=NUM_WORKERS)
valid_dataloader = DataLoader(valid_dataset, batch_size=16, shuffle=False, num_workers=NUM_WORKERS)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=NUM_WORKERS)

In [39]:
class SegmentationBackbone(nn.Module):

    def __init__(self, architecture, encoder, encoder_weights, in_channels, out_channels, **kwargs):
        super().__init__()
        self.model = smp.create_model(architecture, encoder_name=encoder, encoder_weights=encoder_weights,
            in_channels=in_channels, classes=out_channels, **kwargs)


    def forward(self, image):
        mask = self.model(image)
        return mask


In [40]:
def train(loader, model, optimizer, loss_fn, scaler):
    loop = tqdm(loader)

    for batch_idx, (data, targets) in enumerate(loop):
        data = data.to(device=DEVICE)
        targets = targets.float().unsqueeze(1).to(device=DEVICE)

        # forward pass
        with torch.cuda.amp.autocast():
            predictions = model(data)
            loss = loss_fn(predictions, targets)

        # backward pass
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # update tqdm loop
        loop.set_postfix(loss=loss.item())

In [41]:
def main():
    model = SegmentationBackbone(architecture="UNET", encoder="resnet34", encoder_weights="imagenet", 
    in_channels=3, out_channels=1)
    loss_fn = smp.losses.DiceLoss(smp.losses.BINARY_MODE, from_logits=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    scaler = torch.cuda.amp.GradScaler()
    for epoch in range(NUM_EPOCHS):
        train(train_dataloader, model, optimizer, loss_fn, scaler)

        # save the model
        checkpoint = {
            "state_dict": model.state_dict(),
            "optimizer": optimizer.state_dict(),
        }
        utils.save_checkpoint(checkpoint)

        # print accuracy
        utils.check_accuracy(valid_dataloader, model, device=DEVICE)

if __name__ == "__main__":
    main()

  0%|          | 0/212 [00:00<?, ?it/s]


TypeError: Caught TypeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/tmp/miniconda3/envs/cv/lib/python3.10/site-packages/torch/utils/data/_utils/worker.py", line 287, in _worker_loop
    data = fetcher.fetch(index)
  File "/tmp/miniconda3/envs/cv/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 49, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "/tmp/miniconda3/envs/cv/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 49, in <listcomp>
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "/tmp/miniconda3/envs/cv/lib/python3.10/site-packages/torch/utils/data/dataset.py", line 471, in __getitem__
    return self.dataset[self.indices[idx]]
  File "/tmp/miniconda3/envs/cv/lib/python3.10/site-packages/torchvision/datasets/oxford_iiit_pet.py", line 110, in __getitem__
    image, target = self.transforms(image, target)
TypeError: Compose.__call__() takes 2 po